# Notebook 4: The Sampling Distribution & Unbiasedness of OLS Estimates ($E[\hat{\beta}] = \beta$)
**Prepared for:** Assumptions of Least Square Regressions Lecture Suite  
**Author:** Nick Lim

One of the foundational properties of Ordinary Least Squares (OLS) under the Gauss-Markov assumptions is **Unbiasedness**. Mathematically, $E[\hat{\beta}_0] = \beta_0$ and $E[\hat{\beta}_1] = \beta_1$.

What does **unbiasedness** actually mean in practice? It means that if we repeatedly draw random samples of size $n$ from a population and estimate the regression line for each sample, the **average of our estimated intercepts ($\hat{\beta}_0$) across all samples will exactly equal the true population intercept ($\beta_0$)**, and the **average of our estimated slopes ($\hat{\beta}_1$) will exactly equal the true population slope ($\beta_1$)**.

In this notebook, we will prove this empirically:
1. Create a true **Population of 1,000 points** with the exact relationship: $$Y_i = 1.0 + 2.0 X_i + \epsilon_i, \quad \epsilon_i \sim \mathcal{N}(0, 1), \quad X_i \in [0, 10]$$
2. Run **1,000 independent simulations**: in each loop, draw a random sample of just $n = 30$ points from the population, fit an OLS model, and record $\hat{\beta}_0$ and $\hat{\beta}_1$.
3. Plot histograms of our 1,000 estimated intercepts and slopes to verify that their distributions center directly on $\beta_0 = 1.0$ and $\beta_1 = 2.0$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Polyfill for matplotlib compatibility
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = lambda key: matplotlib.rcParams[key]

# Set random seed for exact reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 12

## 1. Generating the True Population ($N = 1,000$)

Let's create our underlying population of 1,000 observations where:
- True Intercept $\beta_0 = 1.0$
- True Slope $\beta_1 = 2.0$
- Standard Normal Noise $\epsilon_i \sim \mathcal{N}(0, 1)$

In [ ]:
N_pop = 1000
x_pop = np.random.uniform(0, 10, N_pop)
err_pop = np.random.normal(0, 1.0, N_pop)
y_pop = 1.0 + 2.0 * x_pop + err_pop

pop_df = pd.DataFrame({"x": x_pop, "y": y_pop})

plt.figure(figsize=(9, 5.5))
plt.scatter(pop_df["x"], pop_df["y"], color="#3b82f6", alpha=0.5, s=25, label="Population Observations ($N=1,000$)")
x_grid = np.linspace(0, 10, 100)
plt.plot(x_grid, 1.0 + 2.0 * x_grid, color="#1e293b", linewidth=3.5, label="True Population Line ($Y = 1 + 2X$)")
plt.title("True Population Relationship ($N = 1,000$)")
plt.xlabel("Predictor ($X$)")
plt.ylabel("Outcome ($Y$)")
plt.legend()
plt.show()

## 2. Running 1,000 Monte Carlo Simulations ($n = 30$ per sample)

Now, let's pretend we are researchers who only have the budget to collect $n = 30$ data points from this population. Any single sample of 30 points will yield slightly different estimates ($\hat{\beta}_0, \hat{\beta}_1$) due to random sampling variability.

We will repeat this sampling experiment **1,000 times** and store the estimated intercept and slope from each loop.

In [ ]:
n_sims = 1000
sample_size = 30

sim_beta0 = []
sim_beta1 = []

for _ in range(n_sims):
    # Randomly sample 30 indices without replacement from our population of 1,000
    sample_indices = np.random.choice(N_pop, size=sample_size, replace=False)
    x_sample = x_pop[sample_indices]
    y_sample = y_pop[sample_indices]
    
    # Fit Ordinary Least Squares
    X_matrix = sm.add_constant(x_sample)
    mod = sm.OLS(y_sample, X_matrix).fit()
    
    sim_beta0.append(mod.params[0])
    sim_beta1.append(mod.params[1])

mean_beta0 = np.mean(sim_beta0)
mean_beta1 = np.mean(sim_beta1)

print("=== Sampling Distribution Summary (Across 1,000 Simulations of n=30) ===")
print(f"True Population Intercept (beta_0)      : 1.0000")
print(f"Mean Estimated Intercept (mean(hat_b0)) : {mean_beta0:.4f}")
print(f"-> Intercept Bias                       : {mean_beta0 - 1.0:.4f}
")
print(f"True Population Slope (beta_1)          : 2.0000")
print(f"Mean Estimated Slope (mean(hat_b1))     : {mean_beta1:.4f}")
print(f"-> Slope Bias                           : {mean_beta1 - 2.0:.4f}")

## 3. Visualizing Unbiasedness: Histograms of $\hat{\beta}_0$ and $\hat{\beta}_1$

Let's plot the distributions of our 1,000 estimated parameters. We overlay two vertical lines:
- **Dashed Black Line:** The exact true population parameter ($\beta_0 = 1.0$, $\beta_1 = 2.0$).
- **Solid Colored Line:** The average of our 1,000 simulation estimates.

Notice how virtually indistinguishable the two lines are—proving empirically that **OLS estimates are unbiased**!

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Sampling Distribution of Intercept (beta_0)
sns.histplot(sim_beta0, kde=True, color="#f59e0b", ax=axes[0], bins=35, alpha=0.65)
axes[0].axvline(1.0, color="#1e293b", linestyle="--", linewidth=2.8, label="True Parameter ($\beta_0 = 1.0$)")
axes[0].axvline(mean_beta0, color="#dc2626", linestyle="-", linewidth=2.2, label=f"Mean Sample Estimate ({mean_beta0:.3f})")
axes[0].set_title("Sampling Distribution of Intercept ($\hat{\beta}_0$)")
axes[0].set_xlabel("Estimated Intercept ($\hat{\beta}_0$)")
axes[0].set_ylabel("Frequency (out of 1,000 Samples)")
axes[0].legend()

# Right: Sampling Distribution of Slope (beta_1)
sns.histplot(sim_beta1, kde=True, color="#16a34a", ax=axes[1], bins=35, alpha=0.65)
axes[1].axvline(2.0, color="#1e293b", linestyle="--", linewidth=2.8, label="True Parameter ($\beta_1 = 2.0$)")
axes[1].axvline(mean_beta1, color="#dc2626", linestyle="-", linewidth=2.2, label=f"Mean Sample Estimate ({mean_beta1:.3f})")
axes[1].set_title("Sampling Distribution of Slope ($\hat{\beta}_1$)")
axes[1].set_xlabel("Estimated Slope ($\hat{\beta}_1$)")
axes[1].set_ylabel("Frequency (out of 1,000 Samples)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Key Takeaways & Intuition

1. **Unbiasedness is an Average Property:** In any single sample of 30 points, $\hat{\beta}_1$ might turn out to be `1.85` or `2.15` because of random noise. But over many samples, these errors cancel out perfectly around the true slope `2.0`.
2. **Role of Sample Size ($n$):** If we increased our sample size from $n = 30$ to $n = 200$, the estimates $\hat{\beta}_1$ would still center on `2.0` (still unbiased), but the spread (standard error / width of the bell curve) would shrink dramatically, making each individual estimate much more precise (Gauss-Markov efficiency).